In [1]:
library(dplyr)
library(tidyr)
library(data.table)


载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



载入程辑包：‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
cd4 <- read.csv('CD4T_eqtl_meta_results_GWASrisk.csv')
cd8 <- read.csv('CD8T_eqtl_meta_results_GWASrisk.csv')
eg = unique(c(cd4$Gene,cd8$Gene))

In [2]:
length(eg)

[1] 225

In [3]:
eg_df = data.frame(row.names = eg)

# coloc

In [4]:
res = read.csv('all_celltypes_coloc_summary.csv')
res = res[res$PP4 > 0.75,]

In [5]:
coloc_matrix <- res %>%
  select(ct, gene, PP4) %>%
  pivot_wider(
    names_from = ct,
    values_from = PP4,
    values_fill = 0
  )

coloc_matrix <- coloc_matrix %>%
  mutate(
    coloc_score = pmax(CD4T, CD8T)
  )

coloc_matrix = data.frame(coloc_matrix)
rownames(coloc_matrix) <- coloc_matrix$gene

coloc_matrix = coloc_matrix[coloc_matrix$gene %in% eg,]
eg_df$coloc_score = 0
eg_df[rownames(coloc_matrix),'coloc_score'] = coloc_matrix$coloc_score

# Hi-C

In [6]:
h1 = read.csv('CD8T_risk_loop.csv')
h1 = h1[!duplicated(h1$gene),]
h1$ct = 'CD8T'
h2 = read.csv('CD4T_risk_loop.csv')
h2 = h2[!duplicated(h2$gene),]
h2$ct = 'CD4T'

h = rbind(h1,h2)

hic_gene <- h %>%
  distinct(gene, ct) %>%
  mutate(value = 1) %>%
  tidyr::pivot_wider(
    names_from = ct,
    values_from = value,
    values_fill = 0
  )

hic_gene = data.frame(hic_gene)
rownames(hic_gene) = hic_gene$gene
hic_gene = hic_gene[hic_gene$gene %in% eg,]

hic_gene <- hic_gene %>%
  mutate(
    hic_score = (CD4T + CD8T)/2
  )

In [7]:
eg_df$hic_score = 0
eg_df[rownames(hic_gene),'hic_score'] = hic_gene$hic_score

# DEG

In [8]:
deg_df = data.frame(row.names = eg)

In [9]:
ct = 'CD4T'
meta3 <- data.frame(fread(paste0(ct,'_eqtl_meta_results_GWASrisk.csv')))

pbmc <- readRDS(paste0(ct,'_disease_DEG_new.rds'))
LN <- read.csv(paste0(ct,'_DEG_LD.csv'),row.names = 1)


logFC_cutoff <- 0.5
pval_cutoff <- 0.05

pbmc = pbmc[rownames(pbmc) %in% meta3$Gene & pbmc$p_val_adj< pval_cutoff & pbmc$avg_log2FC > logFC_cutoff ,]
LN = LN[rownames(LN) %in% meta3$Gene & LN$p_val_adj < pval_cutoff & LN$avg_log2FC > logFC_cutoff ,]

pbmc_cd4 <- pbmc %>%
  mutate(
    rank_score = 0
  )

n <- nrow(pbmc_cd4)
pbmc_cd4$rank_score <-
  rank(
    pbmc_cd4$avg_log2FC,
    ties.method = "average"
  )


pbmc_cd4$rank_score <-
  (pbmc_cd4$rank_score - 0.5) / n

deg_df[rownames(pbmc_cd4),'cd4_pbmc'] = pbmc_cd4$rank_score 

LN_cd4 <- LN %>%
  mutate(
    rank_score = 0
  )

n <- nrow(LN_cd4)

LN_cd4$rank_score <-
  rank(
    LN_cd4$avg_log2FC,
    ties.method = "average"
  )


LN_cd4$rank_score <-
  (LN_cd4$rank_score - 0.5) / n

deg_df[rownames(LN_cd4),'cd4_LN'] = LN_cd4$rank_score 

In [10]:
ct = 'CD8T'
meta3 <- data.frame(fread(paste0(ct,'_eqtl_meta_results_GWASrisk.csv')))

pbmc <- readRDS(paste0(ct,'_disease_DEG_new.rds'))
LN <- read.csv(paste0(ct,'_DEG_LD.csv'),row.names = 1)

logFC_cutoff <- 0.5
pval_cutoff <- 0.05

pbmc = pbmc[rownames(pbmc) %in% meta3$Gene & pbmc$p_val_adj< pval_cutoff & pbmc$avg_log2FC > logFC_cutoff ,]
LN = LN[rownames(LN) %in% meta3$Gene & LN$p_val_adj < pval_cutoff & LN$avg_log2FC > logFC_cutoff ,]

pbmc_cd4 <- pbmc %>%
  mutate(
    rank_score = 0
  )

n <- nrow(pbmc_cd4)

pbmc_cd4$rank_score <-
  rank(
    pbmc_cd4$avg_log2FC,
    ties.method = "average"
  )


pbmc_cd4$rank_score <-
  (pbmc_cd4$rank_score - 0.5) / n

deg_df[rownames(pbmc_cd4),'cd8_pbmc'] = pbmc_cd4$rank_score 

LN_cd4 <- LN %>%
  mutate(
    rank_score = 0
  )

n <- nrow(LN_cd4)

LN_cd4$rank_score <-
  rank(
    LN_cd4$avg_log2FC,
    ties.method = "average"
  )


LN_cd4$rank_score <-
  (LN_cd4$rank_score - 0.5) / n

deg_df[rownames(LN_cd4),'cd8_LN'] = LN_cd4$rank_score 

In [11]:
deg_df[is.na(deg_df)] = 0

all_deg_score <- deg_df %>%
  mutate(
    PBMC_score = rowMeans(
      cbind(cd4_pbmc,cd8_pbmc),
      na.rm=TRUE
    ),
    PLN_score = rowMeans(
      cbind(cd4_LN,cd8_LN),
      na.rm=TRUE
    ),
    Tx_score = rowMeans(
      cbind(PBMC_score, PLN_score),
      na.rm=TRUE
    )
  )
eg_df$deg_score = all_deg_score[rownames(eg_df),]$Tx_score

# score

In [12]:
convert_to_plike <- function(score){

  p <- rep(1, length(score))

  idx <- score > 0

  n <- sum(idx)

  if(n > 0){

    p[idx] <-
      (rank(
        -score[idx],
        ties.method="average"
      ) - 0.5) / n

  }

  return(p)
}

In [13]:
eg_df2 <- eg_df %>%
  mutate(
    P_genetic = convert_to_plike(coloc_score),
    P_HiC = convert_to_plike(hic_score),
    P_Tx = convert_to_plike(deg_score)
  )

eg_df2 <- eg_df2 %>%
  mutate(
    Fisher_score =
      -2 * (
        log(P_genetic) +
        log(P_HiC) +
        log(P_Tx)
      )
  )

eg_df2 <- eg_df2 %>%
  mutate(
    Integrated_score =
      percent_rank(Fisher_score)
  )

ed_df2 = eg_df2[order(eg_df2$Integrated_score,decreasing = T),]

gene_weight <- ed_df2  %>%
  mutate(
    weight = Integrated_score / sum(Integrated_score)
  )

In [14]:
ed_df2 = eg_df2[order(eg_df2$Integrated_score,decreasing = T),]

In [15]:
gene_weight <- ed_df2  %>%
  mutate(
    weight = Integrated_score / sum(Integrated_score)
  )

In [16]:
write.csv(gene_weight,'eGene_prioritization.csv')